# Delta Lake com Apache Spark

Este notebook demonstra o uso do **Delta Lake** integrado ao **Apache Spark (PySpark)**.

## Cenário
Tabela de **clientes** de uma loja fictícia, com operações de INSERT, UPDATE e DELETE usando o formato Delta.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Delta Lake Demo") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession iniciada com sucesso!")

SparkSession iniciada com sucesso!


## Modelo da Tabela

Tabela `clientes` com os campos:
- `id`: identificador único
- `nome`: nome do cliente
- `cidade`: cidade do cliente
- `valor_compra`: valor total de compras

## DDL — Definição da Tabela

```sql
CREATE TABLE clientes (
    id INT,
    nome STRING,
    cidade STRING,
    valor_compra DOUBLE
) USING DELTA;

## Diagrama ER

```
┌─────────────────────────┐
│        CLIENTES         │
├─────────────────────────┤
│ PK  id           INT    │
│     nome         STRING │
│     cidade       STRING │
│     valor_compra DOUBLE │
└─────────────────────────┘
```

In [2]:
from pyspark.sql import Row

# Dados iniciais
dados = [
    Row(id=1, nome="Ana Silva", cidade="São Paulo", valor_compra=1500.0),
    Row(id=2, nome="Bruno Costa", cidade="Rio de Janeiro", valor_compra=2300.0),
    Row(id=3, nome="Carla Souza", cidade="Curitiba", valor_compra=800.0),
    Row(id=4, nome="Diego Lima", cidade="Belo Horizonte", valor_compra=3200.0),
    Row(id=5, nome="Eva Martins", cidade="Porto Alegre", valor_compra=950.0),
]

df = spark.createDataFrame(dados)

# Salvar como tabela Delta
df.write.format("delta").mode("overwrite").save("/home/jovyan/work/data/clientes_delta")

print("Tabela Delta criada com sucesso!")
df.show()

Tabela Delta criada com sucesso!
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  1|  Ana Silva|     São Paulo|      1500.0|
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  3|Carla Souza|      Curitiba|       800.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
+---+-----------+--------------+------------+



## UPDATE — Atualizando registros

Atualizando a cidade e valor de compra da cliente Ana Silva.

In [3]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/home/jovyan/work/data/clientes_delta")

deltaTable.update(
    condition="id = 1",
    set={"cidade": "'Campinas'", "valor_compra": "1800.0"}
)

print("UPDATE realizado com sucesso!")
deltaTable.toDF().show()

UPDATE realizado com sucesso!
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
|  3|Carla Souza|      Curitiba|       800.0|
|  1|  Ana Silva|      Campinas|      1800.0|
+---+-----------+--------------+------------+



## DELETE — Removendo registros

Removendo o cliente com id = 3 (Carla Souza).

In [4]:
deltaTable.delete(condition="id = 3")

print("DELETE realizado com sucesso!")
deltaTable.toDF().show()

DELETE realizado com sucesso!
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
|  1|  Ana Silva|      Campinas|      1800.0|
+---+-----------+--------------+------------+



## Histórico de alterações

O Delta Lake mantém um histórico completo de todas as operações realizadas na tabela (time travel).

In [5]:
historico = deltaTable.history()
historico.select("version", "timestamp", "operation").show(truncate=False)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-06-02 19:09:55.617|DELETE   |
|1      |2026-06-02 19:09:26.548|UPDATE   |
|0      |2026-06-02 19:07:06.572|WRITE    |
+-------+-----------------------+---------+



## Time Travel — Consultando versões anteriores

Uma das principais vantagens do Delta Lake é poder consultar versões anteriores da tabela.

In [6]:
# Consultando a versão original (antes de qualquer alteração)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("/home/jovyan/work/data/clientes_delta")
print("Versão 0 (original):")
df_v0.show()

# Consultando após o UPDATE
df_v1 = spark.read.format("delta").option("versionAsOf", 1).load("/home/jovyan/work/data/clientes_delta")
print("Versão 1 (após UPDATE):")
df_v1.show()

Versão 0 (original):
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
|  1|  Ana Silva|     São Paulo|      1500.0|
|  3|Carla Souza|      Curitiba|       800.0|
+---+-----------+--------------+------------+

Versão 1 (após UPDATE):
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
|  3|Carla Souza|      Curitiba|       800.0|
|  1|  Ana Silva|      Campinas|      1800.0|
+---+-----------+--------------+------------+



## Conclusão

O Delta Lake adiciona ao Apache Spark:
- Transações ACID
- Versionamento de dados (Time Travel)
- Operações de UPDATE e DELETE (não disponíveis no parquet comum)
- Histórico completo de alterações